# Post-Deploy App Tags

Applies bundle preset tags to the Databricks App after deployment.

The DABs app resource YAML schema (`resources.apps`) does not support a `tags` property
(`additionalProperties: false`), so tags cannot be declared in `zerobus_ingest.app.yml`.
This notebook bridges that gap by calling the **Workspace Entity Tag Assignments API**
to apply the same tags that other bundle resources receive via `presets.tags`.

**Usage:**
- **As a job task:** `databricks bundle run post_deploy_app_tags --target dev`
- **Interactively:** Run cells in order; parameters default to dev-target values

The job resource at `resources/post_deploy_app_tags.job.yml` passes all tag values
as `base_parameters`, sourced from the bundle's `${var.*}` tag variables.

In [0]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

In [0]:
# ── Widget definitions (populated by job parameters or manual entry) ──────

dbutils.widgets.text("app_name", "dev-dbxw-0bus-ingest", "App Name")
dbutils.widgets.text("tag_project", "dbxWearables ZeroBus", "project")
dbutils.widgets.text("tag_businessUnit", "Healthcare and Life Sciences", "businessUnit")
dbutils.widgets.text("tag_developer", "matthew.giglia@databricks.com", "developer")
dbutils.widgets.text("tag_requestedBy", "Healthcare Providers and Health Plans", "requestedBy")
dbutils.widgets.text("tag_RemoveAfter", "2027-03-04", "RemoveAfter")
dbutils.widgets.text("tag_env", "dev", "env")

# ── Read parameter values ─────────────────────────────────────────────────

app_name = dbutils.widgets.get("app_name")
tag_project = dbutils.widgets.get("tag_project")
tag_business_unit = dbutils.widgets.get("tag_businessUnit")
tag_developer = dbutils.widgets.get("tag_developer")
tag_requested_by = dbutils.widgets.get("tag_requestedBy")
tag_remove_after = dbutils.widgets.get("tag_RemoveAfter")
tag_env = dbutils.widgets.get("tag_env")

# ── Build tags dict ───────────────────────────────────────────────────────

tags = {
    "project": tag_project,
    "businessUnit": tag_business_unit,
    "developer": tag_developer,
    "requestedBy": tag_requested_by,
    "RemoveAfter": tag_remove_after,
    "env": tag_env,
}

print(f"App name: {app_name}")
print(f"Tags to apply ({len(tags)}):")
for k, v in tags.items():
    print(f"  {k} = {v}")

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# ── Verify the app exists ─────────────────────────────────────────────────

app = w.apps.get(name=app_name)
print(f"App found: {app.name} (status: {app.app_status})")

# ── Apply tags via Workspace Entity Tag Assignments API ───────────────────

successes = 0
failures = 0
errors = []

for key, value in tags.items():
    try:
        w.workspace_entity_tag_assignments.create_tag_assignment(
            entity_type="apps",
            entity_id=app_name,
            tag_key=key,
            tag_value=value,
        )
        successes += 1
        print(f"  [OK] {key} = {value}")
    except Exception as e:
        failures += 1
        error_msg = str(e)
        errors.append((key, error_msg))
        print(f"  [FAIL] {key} = {value} -- {error_msg}")

# ── Summary ───────────────────────────────────────────────────────────────

total = successes + failures
print(f"\n{successes}/{total} tags applied to '{app_name}'")

if failures > 0 and failures == total:
    # All tags failed — likely an entity_type issue
    unique_errors = set(e for _, e in errors)
    if len(unique_errors) == 1:
        print(
            "\nNOTE: All tags failed with the same error. The entity_type "
            "value 'apps' may not be recognized yet. Try 'databricks_apps' "
            "or check the Workspace Entity Tag Assignments API docs — app "
            "tagging is in Public Preview and the entity_type enum may differ."
        )

if failures > 0:
    raise Exception(f"{failures}/{total} tag assignments failed for '{app_name}'")

In [0]:
# ── List tags currently assigned to the app ───────────────────────────────

try:
    tags_result = w.workspace_entity_tag_assignments.list_tag_assignments(
        entity_type="apps",
        entity_id=app_name,
    )

    assigned_tags = list(tags_result)
    if assigned_tags:
        print(f"Tags on '{app_name}' ({len(assigned_tags)}):")
        for tag in assigned_tags:
            print(f"  {tag.tag_key} = {tag.tag_value}")
    else:
        print(f"No tags found on '{app_name}'")

except Exception as e:
    print(f"Could not list tags for '{app_name}': {e}")
    print(
        "This may be expected if the Entity Tag Assignments API "
        "does not yet support 'apps' as an entity_type."
    )